# Benchmark: Key-Filtered vs Brute-Force Enumeration

Loads pre-computed data from `benchmark_scaling.csv` (generated by `run_benchmark_scaling.py`)
and plots pairs/s as a function of number of reaction templates, with ±1 std shaded band.

## Reproducing the data

The CSV consumed by this notebook lives under `data/paper/`, which is **not checked into git**. Regenerate it with:

```bash
pixi run -e dev python scripts/run_benchmark_scaling.py
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_FILE = "../../data/paper/benchmark_scaling.csv"

In [ ]:
df = pd.read_csv(DATA_FILE)

xs     = df["n_templates"].to_numpy()
mean_f = df["filtered_mean"].to_numpy()
std_f  = df["filtered_std"].to_numpy()
mean_b = df["brute_mean"].to_numpy()
std_b  = df["brute_std"].to_numpy()

In [ ]:
from pathlib import Path

color_f = "#2196F3"  # blue
color_b = "#F44336"  # red

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(xs, mean_f, color=color_f, marker="o", markersize=2, linewidth=1, label="Filtered by functional groups")
ax.fill_between(xs, mean_f - std_f, mean_f + std_f, color=color_f, alpha=0.2)

ax.plot(xs, mean_b, color=color_b, marker="s", markersize=2, linewidth=1, label="Not filtered")
ax.fill_between(xs, mean_b - std_b, mean_b + std_b, color=color_b, alpha=0.2)

ax.set_xlabel("Number of reaction SMARTS", fontsize=15)
ax.set_ylabel("Pairs / second", fontsize=15)
ax.tick_params(axis="both", labelsize=13)
ax.legend(fontsize=13)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

out = Path("../../smartreact_paper/figures/filtering.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=300, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

## Summary table

In [ ]:
pd.DataFrame({
    "n_templates":   xs,
    "filtered_mean": mean_f.round(1),
    "filtered_std":  std_f.round(1),
    "brute_mean":    mean_b.round(1),
    "brute_std":     std_b.round(1),
    "speedup":       (mean_f / mean_b).round(2),
})